# CANDIDATE FEATURE IDENTIFICATION
The purpose of this notebook is to aggregate in one location the potential features to include in the training set. 
Inputs: External Dataset Parquet Files. 
Desired Outputs: 
- Of redundant variables, select the most reliable and informative one. 
- Understand how to calculate any computed features.

Notes
- Use a separate .py module under src/external_dataset_name/ingest.py to pull in the data with API or other method and write to data/interim/external_dataset_name.parquet. 
- This notebook should read in just a small sample of each dataset for feature exploration. Reading in the full dataset may crash the Jupyter kernel. 

For each feature, attempt to identify:
- time of observation
- unit of observation / what it actually is
- geographic scope (i.e. is it at the census block group, tract, or zip code level?)
- missingness
- what preprocessing is required (e.g. geocoding to census block group, converting a raw count to density)

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
pd.set_option('display.max_columns', None)

## Lottery and Sales Data (for reference when merging)

In [3]:
lottery_locations_file_path = Path(".").resolve().parent / "data/final/location_data.parquet"

#locations_pq = pq.ParquetFile(lottery_locations_file_path)
#locations_first_100_rows = next(locations_pq.iter_batches(batch_size=100))
#locations_df = pa.Table.from_batches([locations_first_100_rows]).to_pandas()

locations_df = pd.read_parquet(str(lottery_locations_file_path))

In [4]:
locations_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid
2695,109346,SEEMA NOREEN,29-24A HOYT AVENUE SOUTH,ASTORIA,11102,0,GROCERY STORE,-73.91798,40.77064,b'\x01\x01\x00\x00\x00t\x98//\xc0zR\xc0\xa6~\x...,360810125002,36081012500
12382,109945,7-ELEVEN #17750D,3 RAILROAD AVE,ROSLYN,11576,30,CONV. STORE,-73.64355,40.79316,b'\x01\x01\x00\x00\x00\xd0\xd5V\xec/iR\xc0h?RD...,360593020002,36059302000
11689,108029,MOUNTAIN VIEW RESTAURANT,241 STONE SCHOOL HOUSE RD,BLOOMINGBURG,12721,0,RESTAURANT-HOTEL,-74.40151,41.56800,b'\x01\x01\x00\x00\x00\x16\xc1\xffV\xb2\x99R\x...,360710109022,36071010902
5419,49020,NEWCOMB OIL CO INC,337 ROUTE 31,MACEDON,14502,0,CONV. STORE,-77.36058,43.06466,"b'\x01\x01\x00\x00\x00\xde\xe5""\xbe\x13WS\xc0\...",361170203031,36117020303
7387,75859,C & K THE CORNER STORE,630 COLVIN AVE,BUFFALO,14216,0,GROCERY STORE,-78.85921,42.95813,b'\x01\x01\x00\x00\x00[\x99\xf0K\xfd\xb6S\xc0\...,360290051003,36029005100


In [5]:
lottery_sales_file_path = Path(".").resolve().parent / "data/final/sales_data.parquet"

#sales_pq = pq.ParquetFile(lottery_sales_file_path)
#sales_first_100_rows = next(sales_pq.iter_batches(batch_size=1000))
#sales_df = pa.Table.from_batches([sales_first_100_rows]).to_pandas()

sales_df = pd.read_parquet(str(lottery_sales_file_path))

In [6]:
sales_df.sample(5)

,agtno,total_sales,first_date,last_date,first_pos_date,last_pos_date,pos_sales_days,num_cal_days,num_active_days,days_before_first_sale,days_after_last_sale,avg_cal_daily_sales,avg_active_daily_sales,use_active_period,avg_daily_sales
11456,73733,194804.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,533.709589,533.709589,False,533.709589
2207,107470,922287.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,2526.815068,2526.815068,False,2526.815068
1227,104409,269655.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,359.0,365,365.0,0.0,0.0,738.782192,738.782192,False,738.782192
9762,50993,11047815.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,30267.986301,30267.986301,False,30267.986301
1081,103965,1406866.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,3854.428767,3854.428767,False,3854.428767


## Overture Maps Data

In [7]:
overture_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_overture.parquet"

overture_pq = pq.ParquetFile(overture_file_path)
overture_first_100_rows = next(overture_pq.iter_batches(batch_size=100))
overture_df = pa.Table.from_batches([overture_first_100_rows]).to_pandas()

In [8]:
overture_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,gas_station_within_0.5_mi,gas_station_within_1_mi,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,shopping_within_0.5_mi,shopping_within_1_mi,social_or_community_service_within_0.5_mi,social_or_community_service_within_1_mi,restaurant_within_0.5_mi,restaurant_within_1_mi,bar_within_0.5_mi,bar_within_1_mi
0,97637,TURCA GROCERY CORP,1142 E 165TH ST,BRONX,10459,-73.88748,40.82523,b'\x01\x01\x00\x00\x00\xa9\xf6\xe9x\xccxR\xc0\...,360050119005,36005011900,6,13,3,8,2,8,11,42,6,15,2,3
94,114489,LORRIES DELI GROCERY CORP,365 E 193RD ST,BRONX,10458,-73.89090,40.86311,b'\x01\x01\x00\x00\x00i\x00o\x81\x04yR\xc0\xdf...,360050399013,36005039901,4,10,2,12,2,13,11,29,6,23,2,7
90,110788,SUNRISE 99 CENT STORE INC,4008 QUEENS BLVD,SUNNYSIDE,11104,-73.92392,40.74361,b'\x01\x01\x00\x00\x00\xb3\x07Z\x81!{R\xc0A}\x...,360810181013,36081018101,5,14,3,5,3,16,6,38,11,24,9,16
92,76475,QUICKLEES #12,349 W MORRIS ST,BATH,14810,-77.33299,42.33839,b'\x01\x01\x00\x00\x00H\xf9I\xb5OUS\xc0\x8f\xa...,361019611001,36101961100,2,2,0,0,0,0,1,9,0,1,0,2
97,112092,FAMILY CARE PHARMACY,2518 ADAM CLAYTON POWELL JR BL,NEW YORK,10039,-73.93875,40.82223,"b""\x01\x01\x00\x00\x00\xaeG\xe1z\x14|R\xc0!\xe...",360610232004,36061023200,4,13,4,20,2,12,23,87,5,22,2,15


## LEHD Data

In [9]:
rac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_rac.parquet"

rac_pq = pq.ParquetFile(rac_file_path)
rac_first_100_rows = next(rac_pq.iter_batches(batch_size=100))
rac_df = pa.Table.from_batches([rac_first_100_rows]).to_pandas()

In [10]:
rac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,rac_c000,rac_ce01,rac_ce02,rac_ce03
91,110085,US FOOD MART DELI & GROCERY IN,952 CRANE ST,SCHENECTADY,12303,-73.94798,42.79824,b'\x01\x01\x00\x00\x00\xc6PN\xb4\xab|R\xc0\xf5...,360930214002,36093021400,257.0,48.0,86.0,123.0
13,102746,NEW BOHEMIA GAS AND REPAIR INC,4909 SUNRISE HGWY,BOHEMIA,11716,-73.10274,40.75947,b'\x01\x01\x00\x00\x00m\xff\xcaJ\x93FR\xc0\x85...,361031466171,36103146617,1036.0,165.0,225.0,646.0


In [11]:
wac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_wac.parquet"

wac_pq = pq.ParquetFile(wac_file_path)
wac_first_100_rows = next(wac_pq.iter_batches(batch_size=100))
wac_df = pa.Table.from_batches([wac_first_100_rows]).to_pandas()

In [12]:
wac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,wac_c000,wac_cns07,wac_cns18,wac_ce01,wac_ce02,wac_ce03
65,89815,AL'S DELI & GROCERY I CORP,160-26 BAISLEY BLVD,JAMAICA,11434,-73.78143,40.67699,b'\x01\x01\x00\x00\x00@\x87\xf9\xf2\x02rR\xc0\...,360810294001,36081029400,96.0,69.0,3.0,24.0,34.0,38.0
41,78564,MOUNT MART INC,140 W SANDFORD BLVD,MOUNT VERNON,10550,-73.83829,40.89820,b'\x01\x01\x00\x00\x00\x16\xa4\x19\x8b\xa6uR\x...,361190028002,36119002800,67.0,0.0,6.0,16.0,18.0,33.0


## AADT Data

In [13]:
aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"

aadt_pq = pq.ParquetFile(aadt_file_path)
aadt_first_100_rows = next(aadt_pq.iter_batches(batch_size=100))
aadt_df = pa.Table.from_batches([aadt_first_100_rows]).to_pandas()

In [14]:
aadt_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,bg_geoid,tract_geoid,index_right,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,dist_to_read_m
98,108621,SERIE 56 DELI GROCERY INC,1915 LEXINGTON AVE,NEW YORK,10035,-73.94034,40.80037,360610182003,36061018200,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,785.040975
33,114951,HARLEM'S OWN WINE AND LIQUOR L,314 E 106TH ST,NEW YORK,10029,-73.94078,40.78993,360610170004,36061017000,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,1540.238987
48,111369,LUNA LIQUOR CORP,175 E 110TH ST,NEW YORK,10029,-73.94321,40.79430,360610172005,36061017200,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,1306.842823
78,113607,PARK AVENUE COMMUNITY DELI COR,1886 PARK AVE,NEW YORK,10035,-73.93758,40.80727,360610206002,36061020600,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,787.199410
13,50993,WEGMANS #13,2301 LYELL AVE,ROCHESTER,14606,-77.68875,43.16374,360550143021,36055014302,55617.0,75037.0,NY390,12,2:2||||,467.163427


## EPA Data

In [15]:
epa_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_epa.parquet"

epa_pq = pq.ParquetFile(epa_file_path)
epa_first_100_rows = next(epa_pq.iter_batches(batch_size=100))
epa_df = pa.Table.from_batches([epa_first_100_rows]).to_pandas()

In [16]:
epa_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,OBJECTID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,D1A,D1B,D1C,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,NatWalkInd,STATEFP2,COUNTYFP2,TRACTCE2,BLKGRPCE2
39,77604,GEORGIA DELI,272 GEORGIA ST,BUFFALO,14201,0,GROCERY STORE,-78.87996,42.89121,b'\x01\x01\x00\x00\x00qr\xbfCQ\xb8S\xc0 \xefU+...,360290071022,36029007102,155055,36,29,7102,2,14.495834,19.230654,23.504133,0.218530,0.655590,0.315655,9.178267,0.121406,3.836418,9.153986,0.024281,29.815621,16.166667,36,029,007102,2
44,75707,RITE AID #10868,1567 PENFIELD ROAD,ROCHESTER,14625,296,DRUG STORE,-77.49424,43.13234,b'\x01\x01\x00\x00\x00\x01\x18\xcf\xa0\xa1_S\x...,360550116031,36055011603,149749,36,55,11603,1,0.811405,1.165216,5.962885,1.460058,1.648757,2.299768,0.195775,0.275972,0.070762,0.011794,0.000000,15.605053,14.333333,36,055,011603,1
75,77429,RUNWAY #6371,6371 ROUTE 167,DOLGEVILLE,13329,0,CONV. STORE,-74.78544,43.09375,"b'\x01\x01\x00\x00\x00\x18>""\xa6D\xb2R\xc0\x00...",360430110023,36043011002,157376,36,43,11002,3,0.029119,0.042826,0.008076,0.001763,0.000000,0.003071,0.001081,0.001877,0.000000,0.000284,0.000000,2.458854,5.333333,36,043,011002,3
30,88265,CIBAO GROCERY CORP,89-04 86TH ST,WOODHAVEN,11421,0,DELI-GROCERY,-73.85700,40.68939,b'\x01\x01\x00\x00\x00\x02+\x87\x16\xd9vR\xc0@...,360810008001,36081000800,149529,36,81,800,1,21.613733,64.341723,5.358026,0.136221,0.000000,0.544884,0.136221,0.000000,0.000000,0.227035,4.313665,31.268510,13.833333,36,081,000800,1
86,86719,RIVERDALE GOURMET INC,3543 RIVERDALE AVE,BRONX,10463,0,DELI-GROCERY,-73.90703,40.88639,b'\x01\x01\x00\x00\x00k\x9f\x8e\xc7\x0czR\xc0/...,360050295001,36005029500,161694,36,5,29500,1,47.636272,97.838808,16.359932,2.619728,2.619728,0.427711,3.742468,3.795932,0.053464,3.100902,0.000000,33.615015,17.666667,36,005,029500,1


## ACS Data

In [17]:
acs_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_acs.parquet"

acs_pq = pq.ParquetFile(acs_file_path)
acs_first_100_rows = next(acs_pq.iter_batches(batch_size=100))
acs_df = pa.Table.from_batches([acs_first_100_rows]).to_pandas()

In [18]:
acs_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,median_household_income,households_total,total_population,age_18_29_proportion,age_30_64_proportion,age_65_plus_proportion,poverty_rate,education_25plus_total_proportion,bachelors_degree_proportion,post_grad_degree_proportion,in_labor_force_proportion,unemployed_proportion,owner_occupied_proportion,renter_occupied_proportion,family_households_proportion,nonfamily_multiperson_households_proportion,living_alone_households_proportion
78,79240,LA ROSA WINES & LIQUORS,3906 BROADWAY,NEW YORK,10032,0,LIQUOR,-73.94190,40.83796,b'\x01\x01\x00\x00\x00\x8e\x06\xf0\x16H|R\xc0\...,360610245001,36061024500,53604.0,839.0,2100.0,0.234762,0.414762,0.350476,0.051406,0.850476,0.219524,0.030476,0.464762,0.194762,0.000000,1.000000,0.396901,0.087008,0.516091
29,106093,QUICK START CONVENIENCE,465 ROUTE 17M,MIDDLETOWN,10940,0,CONV. STORE,-74.41335,41.46407,b'\x01\x01\x00\x00\x00M\xf3\x8eSt\x9aR\xc0\x01...,360710113013,36071011301,NaN,447.0,1124.0,0.153915,0.699288,0.146797,0.057037,0.886121,0.095196,0.076512,0.756228,0.000000,0.744966,0.255034,0.738255,0.022371,0.239374
35,88265,CIBAO GROCERY CORP,89-04 86TH ST,WOODHAVEN,11421,0,DELI-GROCERY,-73.85700,40.68939,b'\x01\x01\x00\x00\x00\x02+\x87\x16\xd9vR\xc0@...,360810008001,36081000800,112336.0,631.0,1840.0,0.259239,0.504891,0.235870,0.054054,0.891304,0.168478,0.035326,0.530230,0.028892,0.587956,0.412044,0.819334,0.115689,0.064976
17,101551,CHECK CHANGERS NY LLC,218-56 HEMPSTEAD AVE,QUEENS VILLAGE,11429,299,CHECK CASHERS,-73.73611,40.71338,b'\x01\x01\x00\x00\x00\x8c\x10\x1em\x1coR\xc0l...,360810540003,36081054000,107644.0,576.0,1596.0,0.184211,0.604010,0.211779,0.143172,0.831454,0.088972,0.024436,0.580704,0.042476,0.383681,0.616319,0.772569,0.203125,0.024306
53,75707,RITE AID #10868,1567 PENFIELD ROAD,ROCHESTER,14625,296,DRUG STORE,-77.49424,43.13234,b'\x01\x01\x00\x00\x00\x01\x18\xcf\xa0\xa1_S\x...,360550116031,36055011603,35833.0,680.0,1055.0,0.244550,0.542180,0.213270,0.345550,0.841706,0.176303,0.019905,0.610122,0.014058,0.573529,0.426471,0.388235,0.170588,0.441176


## Master Data Sample

In [19]:
master_file_path = Path(".").resolve().parent / "data/final/master_data.parquet"
master_df = pd.read_parquet(str(master_file_path))

master_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,dist_to_read_m,median_household_income,households_total,total_population,age_18_29_proportion,age_30_64_proportion,age_65_plus_proportion,poverty_rate,education_25plus_total_proportion,bachelors_degree_proportion,post_grad_degree_proportion,in_labor_force_proportion,unemployed_proportion,owner_occupied_proportion,renter_occupied_proportion,family_households_proportion,nonfamily_multiperson_households_proportion,living_alone_households_proportion,D1A,D1B,D1C,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,NatWalkInd,gas_station_within_0.5_mi,gas_station_within_1_mi,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,shopping_within_0.5_mi,shopping_within_1_mi,social_or_community_service_within_0.5_mi,social_or_community_service_within_1_mi,restaurant_within_0.5_mi,restaurant_within_1_mi,bar_within_0.5_mi,bar_within_1_mi,rac_c000,rac_ce01,rac_ce02,rac_ce03,wac_c000,wac_cns07,wac_cns18,wac_ce01,wac_ce02,wac_ce03
9796,101354,561 STRAIGHT DELI GROCERY INC,561 LENOX AVE,NEW YORK,10037,0,DELI-GROCERY,-73.93921,40.81648,b'\x01\x01\x00\x00\x00\xe0\x84B\x04\x1c|R\xc0F...,360610230002,36061023000,4366.0,WALTON AVE,19,1:0||||,1115.241106,NaN,832.0,1466.0,0.075716,0.644611,0.279673,0.328610,0.952933,0.217599,0.086630,0.501307,0.079739,0.080529,0.919471,0.296875,0.039663,0.663462,102.423704,183.098635,14.592125,0.557661,1.858869,0.000000,4.461287,3.438908,0.000000,4.275400,0.000000,32.994167,12.833333,4.0,16.0,5.0,27.0,5.0,16.0,28.0,118.0,4.0,29.0,4.0,17.0,658.0,119.0,168.0,371.0,420.0,9.0,99.0,89.0,173.0,158.0
3619,15003,BAGELS & BUNS LTD,2491 VICTORY BLVD,STATEN ISLAND,10314,0,SPECIALTY (FOOD),-74.14627,40.61055,b'\x01\x01\x00\x00\x00\xb3\x98\xd8|\\\x89R\xc0...,360850189011,36085018901,7117.0,NY440-VICTORY B,12,1:0,669.250777,136875.0,719.0,1816.0,0.079846,0.692181,0.227974,0.021704,0.925661,0.292401,0.107930,0.557421,0.005417,0.853964,0.146036,0.910987,0.000000,0.089013,9.653467,23.889576,2.535413,0.645664,0.141731,0.204723,0.511807,0.456689,0.015748,0.559051,0.000000,27.973839,17.500000,1.0,6.0,0.0,1.0,0.0,3.0,1.0,6.0,0.0,5.0,0.0,1.0,1120.0,176.0,221.0,723.0,186.0,19.0,29.0,50.0,61.0,75.0
5227,47710,RYANS WINES & SPIRITS,73 EASTERN BLVD,CANANDAIGUA,14424,0,LIQUOR,-77.26443,42.87790,b'\x01\x01\x00\x00\x004\x85\xcek\xecPS\xc0F\x9...,360690511022,36069051102,4803.0,LAKE SHORE DR,17,1:1||||,1231.019135,23182.0,320.0,510.0,0.200000,0.407843,0.392157,0.219820,0.807843,0.015686,0.000000,0.464706,0.023529,0.290625,0.709375,0.375000,0.131250,0.493750,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0,0.0,1.0,0.0,1.0,0.0,5.0,2.0,2.0,0.0,0.0,333.0,65.0,89.0,179.0,587.0,188.0,123.0,154.0,190.0,243.0
15779,115840,7-ELEVEN #15991F,126 NEW YORK AVE,HUNTINGTON,11743,30,CONV. STORE,-73.41761,40.88591,b'\x01\x01\x00\x00\x00\xe2\x1eK\x1f\xbaZR\xc0\...,361031103004,36103110300,NaN,NaN,NaN,NaN,NaN,169952.0,683.0,1474.0,0.126866,0.537992,0.335142,0.010741,0.934871,0.350068,0.232022,0.562415,0.036635,0.923865,0.076135,0.778917,0.046852,0.174231,1.723696,4.109078,3.543336,0.794025,0.205123,0.476415,0.588902,0.863502,0.016542,0.578976,0.019851,15.416998,10.000000,0.0,4.0,0.0,1.0,0.0,1.0,0.0,4.0,1.0,1.0,1.0,2.0,876.0,140.0,154.0,582.0,374.0,108.0,36.0,67.0,114.0,193.0
2438,73967,RITE AID #4672,9062 ERIE RD,ANGOLA,14006,296,DRUG STORE,-79.04919,42.63794,b'\x01\x01\x00\x00\x00\x93R\xd0\xed%\xc3S\xc0\...,360290155032,36029015503,NaN,NaN,NaN,NaN,NaN,54210.0,609.0,1040.0,0.178846,0.515385,0.305769,0.107664,0.895192,0.163462,0.162500,0.586792,0.074528,0.770115,0.229885,0.482759,0.044335,0.472906,0.662609,1.350090,0.250717,0.046761,0.019898,0.023878,0.010944,0.110435,0.000000,0.038801,0.000000,5.243578,7.333333,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,1.0,3.0,552.0,103.0,139.0,310.0,114.0,39.0,8.0,30.0,37.0,47.0
